In [1]:
import pandas as pd
from os.path import join
import matplotlib.pyplot as plt

In [2]:
MASTER_FP = "../.outFiles/analysis/compressed/master.csv"
OUT_DATA_FOLDER_PATH = "../.outFiles/analysis/out_data"
master_df = pd.read_csv(MASTER_FP)

master_df["tx_date"]  = pd.to_datetime(master_df["tx_date"])
master_df["notif_date"]  = pd.to_datetime(master_df["notif_date"])
master_df

C:\Users\fdirham\AppData\Local\Temp\ipykernel_7436\1464853598.py:3: DtypeWarning: Columns (13,17) have mixed types. Specify dtype option on import or set low_memory=False.
  master_df = pd.read_csv(MASTER_FP)


,p_full_name,p_chamber,p_state,p_district,p_party,asset_name,ticker,ticker_location,tx_type,owner,tx_date,notif_date,amount,asset_desc,amount_min,amount_avg,amount_max,doc_id,data_source
0,Alan S Lowenthal,H,California,47,D,E,EP$C,NaN,S,SP,2012-02-27,2012-02-27,1000-15000,NaN,1001.0,8000.5,15000.0,20000945,SELF_HOUSE
1,Alan S Lowenthal,H,California,47,D,E,EP$C,NaN,S,SP,2012-03-20,2012-03-20,1000-15000,NaN,1001.0,8000.5,15000.0,20000945,SELF_HOUSE
2,Alan S Lowenthal,H,California,47,D,KANSAS CITY SOUTHERN,KSU,NaN,P,SP,2012-06-06,2012-06-06,1000-15000,NaN,1001.0,8000.5,15000.0,20000945,SELF_HOUSE
3,Tom Malinowski,H,New Jersey,7,D,BioLife Solutions Inc,BLFSD,NaN,P,NaN,2012-06-19,2021-08-26,1000-15000,NaN,1001.0,8000.5,15000.0,20019374,WATCHER_HOUSE
4,Tammy Duckworth,S,Illinois,NaN,D,PROCTER GAMBLE COMPANY,PG,NaN,S,JT,2012-07-24,2012-10-05,1000-15000,SUBHOLDING OF BROKERAGE 2 USAA 8425,1001.0,8000.5,15000.0,20000923,SELF_HOUSE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54935,Pete Sessions,H,Texas,17,R,US TREASURY BOND,NaN,NaN,P,SP,2024-03-07,2024-03-08,15000-50000,NaN,15001.0,32500.5,50000.0,NaN,CAPITOL_TRADES
54936,William R Keating,H,Massachusetts,9,D,Capital One Financial Corp,COF,US,P,NaN,2024-03-07,2024-03-13,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES
54937,Mike Garcia,H,California,27,R,US TREASURY BOND,NaN,NaN,P,NaN,2024-03-08,2024-03-13,250000-500000,NaN,250001.0,375000.5,500000.0,NaN,CAPITOL_TRADES
54938,Mark E Green,H,Tennessee,7,R,NGL Energy Partners LP,NGL,US,S,NaN,2024-03-11,2024-03-14,15000-50000,NaN,15001.0,32500.5,50000.0,NaN,CAPITOL_TRADES


In [3]:
# Get eligible tickers
mask = master_df["ticker_location"].isna() | (master_df["ticker_location"] == "US")
mask = mask & ~master_df["ticker"].isna()
e_ticker_df = master_df[mask]
e_ticker_df.groupby("ticker").size().sort_values().to_csv("./e_tickers.csv")

e_ticker_df

,p_full_name,p_chamber,p_state,p_district,p_party,asset_name,ticker,ticker_location,tx_type,owner,tx_date,notif_date,amount,asset_desc,amount_min,amount_avg,amount_max,doc_id,data_source
0,Alan S Lowenthal,H,California,47,D,E,EP$C,NaN,S,SP,2012-02-27,2012-02-27,1000-15000,NaN,1001.0,8000.5,15000.0,20000945,SELF_HOUSE
1,Alan S Lowenthal,H,California,47,D,E,EP$C,NaN,S,SP,2012-03-20,2012-03-20,1000-15000,NaN,1001.0,8000.5,15000.0,20000945,SELF_HOUSE
2,Alan S Lowenthal,H,California,47,D,KANSAS CITY SOUTHERN,KSU,NaN,P,SP,2012-06-06,2012-06-06,1000-15000,NaN,1001.0,8000.5,15000.0,20000945,SELF_HOUSE
3,Tom Malinowski,H,New Jersey,7,D,BioLife Solutions Inc,BLFSD,NaN,P,NaN,2012-06-19,2021-08-26,1000-15000,NaN,1001.0,8000.5,15000.0,20019374,WATCHER_HOUSE
4,Tammy Duckworth,S,Illinois,NaN,D,PROCTER GAMBLE COMPANY,PG,NaN,S,JT,2012-07-24,2012-10-05,1000-15000,SUBHOLDING OF BROKERAGE 2 USAA 8425,1001.0,8000.5,15000.0,20000923,SELF_HOUSE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54922,Doug Lamborn,H,Colorado,5,R,NetApp Inc,NTAP,US,S,SP,2024-03-01,2024-03-14,15000-50000,NaN,15001.0,32500.5,50000.0,NaN,CAPITOL_TRADES
54930,Mitch McConnell,S,Kentucky,NaN,R,Wells Fargo & Co,WFC,US,P,SP,2024-03-05,2024-03-19,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES
54932,Pete Sessions,H,Texas,17,R,General Mills Inc,GIS,US,S,NaN,2024-03-07,2024-03-08,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES
54936,William R Keating,H,Massachusetts,9,D,Capital One Financial Corp,COF,US,P,NaN,2024-03-07,2024-03-13,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES


In [12]:
# Get ticker prices
from os.path import exists

def is_str(to_test):
    return isinstance(to_test, str)


# Combine data
PRICE_FOLDER = "../.outFiles/analysis/ticker_prices/"

data = master_df.to_dict(orient="records")
ticker_dates: dict[str, list] = {}
# Get dates per ticker
for row in data:
    ticker:str = row["ticker"]
    if not is_str(ticker):
        continue

    tx_date:pd.Timestamp = row["tx_date"]
    if ticker_dates.get(ticker):
        ticker_dates.get(ticker).append(tx_date)
    else:
        ticker_dates[ticker] = [tx_date]
    continue


price_data = []
latest_price_data = []
# Get prices per ticker
for k in ticker_dates:
    ticker = k
    date_list = ticker_dates[k]
    ticker = ticker.lower()
    
    ticker_file_path = join(PRICE_FOLDER, ticker + ".csv")
    if not exists(ticker_file_path):
        continue

    ticker_df = pd.read_csv(ticker_file_path)
    ticker_df["date"] = pd.to_datetime(ticker_df["Date"])
    
    # Get latest price data
    ticker_df = ticker_df.sort_values("date", ascending=False)
    latest_row = ticker_df.loc[0].to_dict()
    latest_row["ticker"] = ticker.upper()
    latest_price_data.append(latest_row)
    
    mask = ticker_df["date"].isin(date_list)
    
    ticker_df = ticker_df[mask]
    if not ticker_df.size:
        continue

    ticker_df["ticker"] = ticker.upper()
    price_data.append(ticker_df)

base_price_df = pd.concat(price_data)
base_latest_price_df = pd.DataFrame(latest_price_data)

base_price_df

,Date,Close/Last,Volume,Open,High,Low,date,ticker,Adjusted Close
20,02/15/2024,$157.01,6249241.0,$156.30,$157.42,$156.15,2024-02-15,PG,NaN
37,01/23/2024,$153.98,19101560.0,$153.11,$156.40,$152.89,2024-01-23,PG,NaN
39,01/19/2024,$147.57,7848138.0,$148.25,$148.62,$147.31,2024-01-19,PG,NaN
45,01/10/2024,$149.94,8591116.0,$149.35,$150.00,$149.26,2024-01-10,PG,NaN
70,12/04/2023,$152.06,6578318.0,$151.77,$152.5332,$151.66,2023-12-04,PG,NaN
...,...,...,...,...,...,...,...,...,...
48,01/08/2024,56.28,2050410.0,55.53,56.295,55.53,2024-01-08,SCHX,NaN
37,01/24/2024,$205.07,176863.0,$210.00,$210.00,$203.13,2024-01-24,ABG,NaN
37,01/24/2024,$21.51,580791.0,$21.80,$21.99,$21.175,2024-01-24,AGIO,NaN
35,01/26/2024,78.35,5520565.0,78.59,78.685,78.25,2024-01-26,VCLT,NaN


In [13]:
# Format price dfs
price_df = base_price_df.copy(deep=True)
price_df = price_df.drop(columns=["Date", "Adjusted Close"])
price_df.columns = ["close", "volume", "open", "high", "low", "tx_date", "ticker"]


latest_price_df = base_latest_price_df.copy(deep=True)
latest_price_df = latest_price_df.drop(columns=["Date", "Adjusted Close"])
latest_price_df.columns = ["close", "volume", "open", "high", "low", "tx_date", "ticker"]


def get_avg_price(in_row: dict):
    high, low = in_row["high"], in_row["low"]
    high = str(high)
    high = high.removeprefix("$")
    high = float(high)

    low = str(low)
    low = low.removeprefix("$")
    low = float(low)

    return (high + low) / 2

price_df["avg_ticker_price"] = price_df.apply(get_avg_price, axis=1)
latest_price_df["avg_ticker_price"] = latest_price_df.apply(get_avg_price, axis=1)

In [14]:
# Join price df
tmp_price_df = price_df[["ticker", "avg_ticker_price", "tx_date"]]

ticker_df = master_df.merge(tmp_price_df, how="left", on=["ticker","tx_date"])
mask = ~ticker_df["avg_ticker_price"].isna()
ticker_df = ticker_df[mask]

ticker_df

,p_full_name,p_chamber,p_state,p_district,p_party,asset_name,ticker,ticker_location,tx_type,owner,tx_date,notif_date,amount,asset_desc,amount_min,amount_avg,amount_max,doc_id,data_source,avg_ticker_price
141,Susan M Collins,S,Maine,NaN,R,"American International Group, Inc. (NYSE)",AIG,NaN,S (PARTIAL),SP,2014-03-18,2014-03-25,1000-15000,NaN,1001.0,8000.5,15000.0,fa1f4bdc-b23d-4f6c-ab85-75203a9e80a2,WATCHER_SENATE,49.440
142,Susan M Collins,S,Maine,NaN,R,"Salesforce.com, Inc (NYSE)",CRM,NaN,S (PARTIAL),SP,2014-03-19,2014-03-25,1000-15000,NaN,1001.0,8000.5,15000.0,fa1f4bdc-b23d-4f6c-ab85-75203a9e80a2,WATCHER_SENATE,58.860
143,Susan M Collins,S,Maine,NaN,R,The Boeing Company (NYSE),BA,NaN,P,SP,2014-03-19,2014-03-25,1000-15000,NaN,1001.0,8000.5,15000.0,fa1f4bdc-b23d-4f6c-ab85-75203a9e80a2,WATCHER_SENATE,123.285
145,Sheldon Whitehouse,S,Rhode Island,NaN,D,Schlumberger Limited (NYSE),SLB,NaN,P,JT,2014-03-20,2014-04-28,1000-15000,NaN,1001.0,8000.5,15000.0,6af96223-9cce-47b1-bb59-2060a6de5385,WATCHER_SENATE,90.675
146,Susan M Collins,S,Maine,NaN,R,Microsoft Corporation (NASDAQ),MSFT,NaN,P,SP,2014-03-27,2014-04-01,1000-15000,NaN,1001.0,8000.5,15000.0,07b76e11-bd25-44ea-b098-5916b8cacaaa,WATCHER_SENATE,39.655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54922,Doug Lamborn,H,Colorado,5,R,NetApp Inc,NTAP,US,S,SP,2024-03-01,2024-03-14,15000-50000,NaN,15001.0,32500.5,50000.0,NaN,CAPITOL_TRADES,108.705
54930,Mitch McConnell,S,Kentucky,NaN,R,Wells Fargo & Co,WFC,US,P,SP,2024-03-05,2024-03-19,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES,56.265
54932,Pete Sessions,H,Texas,17,R,General Mills Inc,GIS,US,S,NaN,2024-03-07,2024-03-08,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES,64.495
54936,William R Keating,H,Massachusetts,9,D,Capital One Financial Corp,COF,US,P,NaN,2024-03-07,2024-03-13,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES,137.265


In [15]:
# Discriminate options
mask = ticker_df["asset_name"].str.upper().str.contains("OPTION")
mask = mask | ticker_df["asset_desc"].str.upper().str.contains("OPTION")
ticker_df.loc[mask, "is_option"] = True
ticker_df.loc[~mask, "is_option"] = False

ticker_df

,p_full_name,p_chamber,p_state,p_district,p_party,asset_name,ticker,ticker_location,tx_type,owner,...,notif_date,amount,asset_desc,amount_min,amount_avg,amount_max,doc_id,data_source,avg_ticker_price,is_option
141,Susan M Collins,S,Maine,NaN,R,"American International Group, Inc. (NYSE)",AIG,NaN,S (PARTIAL),SP,...,2014-03-25,1000-15000,NaN,1001.0,8000.5,15000.0,fa1f4bdc-b23d-4f6c-ab85-75203a9e80a2,WATCHER_SENATE,49.440,False
142,Susan M Collins,S,Maine,NaN,R,"Salesforce.com, Inc (NYSE)",CRM,NaN,S (PARTIAL),SP,...,2014-03-25,1000-15000,NaN,1001.0,8000.5,15000.0,fa1f4bdc-b23d-4f6c-ab85-75203a9e80a2,WATCHER_SENATE,58.860,False
143,Susan M Collins,S,Maine,NaN,R,The Boeing Company (NYSE),BA,NaN,P,SP,...,2014-03-25,1000-15000,NaN,1001.0,8000.5,15000.0,fa1f4bdc-b23d-4f6c-ab85-75203a9e80a2,WATCHER_SENATE,123.285,False
145,Sheldon Whitehouse,S,Rhode Island,NaN,D,Schlumberger Limited (NYSE),SLB,NaN,P,JT,...,2014-04-28,1000-15000,NaN,1001.0,8000.5,15000.0,6af96223-9cce-47b1-bb59-2060a6de5385,WATCHER_SENATE,90.675,False
146,Susan M Collins,S,Maine,NaN,R,Microsoft Corporation (NASDAQ),MSFT,NaN,P,SP,...,2014-04-01,1000-15000,NaN,1001.0,8000.5,15000.0,07b76e11-bd25-44ea-b098-5916b8cacaaa,WATCHER_SENATE,39.655,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54922,Doug Lamborn,H,Colorado,5,R,NetApp Inc,NTAP,US,S,SP,...,2024-03-14,15000-50000,NaN,15001.0,32500.5,50000.0,NaN,CAPITOL_TRADES,108.705,False
54930,Mitch McConnell,S,Kentucky,NaN,R,Wells Fargo & Co,WFC,US,P,SP,...,2024-03-19,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES,56.265,False
54932,Pete Sessions,H,Texas,17,R,General Mills Inc,GIS,US,S,NaN,...,2024-03-08,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES,64.495,False
54936,William R Keating,H,Massachusetts,9,D,Capital One Financial Corp,COF,US,P,NaN,...,2024-03-13,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES,137.265,False


In [16]:
mask = ticker_df["is_option"]
options_df = ticker_df[mask]
options_amount = options_df["amount_avg"].sum()
print(options_amount)
options_df

15724065.0


,p_full_name,p_chamber,p_state,p_district,p_party,asset_name,ticker,ticker_location,tx_type,owner,...,notif_date,amount,asset_desc,amount_min,amount_avg,amount_max,doc_id,data_source,avg_ticker_price,is_option
2451,Patrick Toomey,S,Pennsylvania,NaN,R,"SPDR S&amp;P 500 ETF <div class=""text-muted"">O...",SPY,NaN,P,SELF,...,2017-05-02,1000-15000,NaN,1001.0,8000.5,15000.0,945dc22a-c776-4beb-aa83-184abaeb4f87,WATCHER_SENATE,239.6150,True
5218,James R Langevin,H,Rhode Island,2,D,SPDR Dow Jones Industrial Average ETF / Call O...,DIA,NaN,P,NaN,...,2020-10-30,1000-15000,NaN,1001.0,8000.5,15000.0,20017595,WATCHER_HOUSE,254.3150,True
5696,James R Langevin,H,Rhode Island,2,D,"Beyond Meat, Inc. / Call Option",BYND,NaN,P,NaN,...,2020-10-30,1000-15000,NaN,1001.0,8000.5,15000.0,20017595,WATCHER_HOUSE,154.6525,True
5952,Roy Blunt,S,Missouri,NaN,R,"The Kraft Heinz Company <div class=""text-muted...",KHC,NaN,S,SP,...,2019-12-28,1000-15000,NaN,1001.0,8000.5,15000.0,7ddbb448-44d1-47b5-8f47-fa2eab111d60,WATCHER_SENATE,30.7650,True
5964,Roy Blunt,S,Missouri,NaN,R,"Mondelez International, Inc. <div class=""text-...",MDLZ,NaN,S,SP,...,2019-12-28,50000-100000,NaN,50001.0,75000.5,100000.0,7ddbb448-44d1-47b5-8f47-fa2eab111d60,WATCHER_SENATE,52.4850,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16856,Tommy Tuberville,S,Alabama,NaN,R,"Alcoa Corporation <div class=""text-muted"">Opti...",AA,NaN,S,JT,...,2021-07-23,15000-50000,NaN,15001.0,32500.5,50000.0,3bdd9340-7c8c-4ed0-898d-dad3835940d2,WATCHER_SENATE,30.0375,True
16862,Tommy Tuberville,S,Alabama,NaN,R,"Alcoa Corporation <div class=""text-muted"">Opti...",AA,NaN,S,JT,...,2021-07-23,15000-50000,NaN,15001.0,32500.5,50000.0,3bdd9340-7c8c-4ed0-898d-dad3835940d2,WATCHER_SENATE,30.0375,True
16949,Blake D Moore,H,Utah,1,R,"Dollar General Corporation - March 19, 2021 $1...",DG,NaN,P,NaN,...,2021-07-17,1000-15000,NaN,1001.0,8000.5,15000.0,20019092,WATCHER_HOUSE,177.5250,True
16976,Nancy Pelosi,H,California,11,D,Microsoft Corporation - Exercised 100 call opt...,MSFT,NaN,P,SELF,...,2021-04-09,1000000-5000000,NaN,1000001.0,3000000.5,5000000.0,20018539,WATCHER_HOUSE,230.9118,True


In [17]:
def display_number(in_number):
    in_number = str(int(in_number))[::-1]

    to_return = ""
    for i, char in enumerate(in_number):
        if i and i % 3 == 0:
            to_return = "," + to_return
            
        to_return  = char + to_return
    return to_return

print(ticker_df["amount_avg"].sum())
# Approximate returns
mask = ticker_df["tx_type"].str.startswith("S")
sell_df = ticker_df[mask]

mask = ticker_df["tx_type"] == "P"
buy_df = ticker_df[mask]

sell_vol = sell_df["amount_avg"].sum()
buy_vol = buy_df["amount_avg"].sum()

print("buy vol", display_number(buy_vol))
print("sell vol", display_number(sell_vol))

profit = sell_vol - buy_vol
print("profit", display_number(profit))

profit_pct = ((sell_vol/buy_vol) * 100) - 100
print("profit percent", profit_pct)

1267654063.5
buy vol 582,711,117
sell vol 684,942,946
profit 102,231,829
profit percent 17.544170090031088


In [18]:
p_sell_df = sell_df.groupby("p_full_name")["amount_avg"].sum() 
p_sell_df = p_sell_df.rename_axis("p_full_name").reset_index()
p_sell_df = p_sell_df.sort_values("amount_avg", ascending=False)
p_sell_df = p_sell_df.reset_index(drop=True)

p_sell_df

,p_full_name,amount_avg
0,Michael T McCaul,121561012.0
1,Josh Gottheimer,102174479.0
2,Ro Khanna,86231204.5
3,Mark E Green,30590551.5
4,Nancy Pelosi,29409012.5
...,...,...
196,Robert P Casey,8000.5
197,Stephanie I Bice,8000.5
198,Stephen F Lynch,8000.5
199,Joseph P Kennedy,8000.5


In [19]:
p_buy_df = buy_df.groupby("p_full_name")["amount_avg"].sum() 
p_buy_df = p_buy_df.rename_axis("p_full_name").reset_index()
p_buy_df = p_buy_df.sort_values("amount_avg", ascending=False)
p_buy_df = p_buy_df.reset_index(drop=True)

p_buy_df

,p_full_name,amount_avg
0,Ro Khanna,101632507.5
1,Josh Gottheimer,96127891.5
2,Michael T McCaul,93748685.5
3,Nancy Pelosi,45375022.5
4,Mark E Green,28774138.5
...,...,...
156,Larry Bucshon,8000.5
157,Seth Moulton,8000.5
158,Robert B Aderholt,8000.5
159,Roger F Wicker,8000.5


In [40]:
mask = ticker_df["p_full_name"] == "Nancy Pelosi"
np_df = ticker_df[mask]

np_df

,p_full_name,p_chamber,p_state,p_district,p_party,asset_name,ticker,ticker_location,tx_type,owner,...,notif_date,amount,asset_desc,amount_min,amount_avg,amount_max,doc_id,data_source,avg_ticker_price,is_option
6694,Nancy Pelosi,H,California,11,D,"Facebook, Inc. - Class A / Exercised 20 call o...",META,NaN,P,SELF,...,2020-02-11,250000-500000,NaN,250001.0,375000.5,500000.0,20015042,WATCHER_HOUSE,221.51000,True
6696,Nancy Pelosi,H,California,11,D,"Facebook, Inc. - Class A / Exercised 30 call o...",META,NaN,P,SELF,...,2020-02-11,250000-500000,NaN,250001.0,375000.5,500000.0,20015042,WATCHER_HOUSE,221.51000,True
6697,Nancy Pelosi,H,California,11,D,"Amazon.com, Inc. / Sold 20 call options with a...",AMZN,NaN,S,SELF,...,2020-02-11,250000-500000,NaN,250001.0,375000.5,500000.0,20015042,WATCHER_HOUSE,93.79025,True
6699,Nancy Pelosi,H,California,11,D,"Amazon.com, Inc. / Exercised 30 call options (...",AMZN,NaN,P,SELF,...,2020-02-11,1000000-5000000,NaN,1000001.0,3000000.5,5000000.0,20015042,WATCHER_HOUSE,93.79025,True
6700,Nancy Pelosi,H,California,11,D,"Amazon.com, Inc. / Sold 20 call options with a...",AMZN,NaN,S,SELF,...,2020-02-11,250000-500000,NaN,250001.0,375000.5,500000.0,20015042,WATCHER_HOUSE,93.79025,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47675,Nancy Pelosi,H,California,11,D,Apple Inc,AAPL,US,P,SP,...,2023-06-23,250000-500000,NaN,250001.0,375000.5,500000.0,NaN,CAPITOL_TRADES,185.15000,False
47678,Nancy Pelosi,H,California,11,D,Microsoft Corp,MSFT,US,P,SP,...,2023-06-23,500000-1000000,NaN,500001.0,750000.5,1000000.0,NaN,CAPITOL_TRADES,343.52000,False
53052,Nancy Pelosi,H,California,11,D,NVIDIA Corporation,NVDA,US,P,SP,...,2023-12-22,1000000-5000000,NaN,1000001.0,3000000.5,5000000.0,NaN,CAPITOL_TRADES,490.12470,False
54636,Nancy Pelosi,H,California,11,D,Palo Alto Networks Inc,PANW,US,P,SP,...,2024-02-26,500000-1000000,NaN,500001.0,750000.5,1000000.0,NaN,CAPITOL_TRADES,371.91500,False


In [111]:
def calculate_success_for_df(in_df: pd.DataFrame, ticker: str):
    mask = in_df["ticker"] == ticker
    tmp_df = in_df[mask]
    tmp_df = tmp_df.sort_values("tx_date", ascending=True)
    data_list = tmp_df.to_dict(orient="records")
    
    is_started = False
    dark_amount = 0 # How much not tracked
    buy_vol = 0 # How much spent buying stocks
    sell_vol = 0 # How much received selling stocks
    realized = 0 # How much g/l realized through sells
    unrealized = 0 # How much g/l unrealized
    total_returns = 0 # realized + unrealized
    tr_pct = 0 # total returns / amount spent
    start_date = 0
    end_date = pd.NaT
    counted_trades = 0
    
    # Go through each row
    buy_units_list: list[tuple[float, float]] = []
    for row in data_list:
        is_buy = row["tx_type"] == "P"
        is_sell = row["tx_type"].startswith("S")
        is_option = row["is_option"]
        tx_amount = row["amount_avg"]
        ticker_price = row["avg_ticker_price"]
        tx_date = row["tx_date"]

        if is_option:
            dark_amount += tx_amount
            continue

        end_date = tx_date

        if not is_started:
            if is_buy:
                is_started = True
                start_date = tx_date
            else:
                dark_amount += tx_amount
                continue
       
        counted_trades += 1
        if is_buy:
            buy_vol += tx_amount 
            stock_units = tx_amount / ticker_price
            buy_units_list.append((stock_units, ticker_price))
        elif is_sell:
            sell_vol += tx_amount
            stock_units = tx_amount / ticker_price
            old_spent = 0

            # Sell as needed from buy units list
            for bu_idx, bu_obj in enumerate(buy_units_list):
                old_su, old_tp = bu_obj
                if old_su == 0:
                    continue
                
                # If more of old stock units than current, then we take all from old and set current to 0
                if old_su >= stock_units:
                    old_spent += stock_units * old_tp
                    
                    # Update old obj in buy unit list
                    n_old_su = old_su - stock_units
                    buy_units_list[bu_idx] = (n_old_su, old_tp)
                
                    stock_units = 0 # Update stock units
                    break
                # If less of old stock units from current, we take all from old until its 0, subtracting from current
                else:
                    stock_units = stock_units - old_su 
                    old_spent += old_su * old_tp
                    
                    # Update old obj in buy unit list
                    buy_units_list[bu_idx] = (0, old_tp)
            
            if stock_units > 0:
                modifier = stock_units * ticker_price
                tx_amount = tx_amount - modifier
            
            gain = tx_amount - old_spent
            realized += gain
        
            buy_units_list = list(filter(lambda x: x[0] > 0, buy_units_list))
        
    # Find unrealized
    unrealized_spent = 0 # How much spent to buy stock units
    unrealized_su = 0 # How many stock units

    for bu_idx, bu_obj in enumerate(buy_units_list):
        old_su, old_tp = bu_obj
        unrealized_spent += old_su * old_tp
        unrealized_su += old_su
    
    mask = latest_price_df["ticker"] == ticker
    latest_price = latest_price_df[mask].iloc[0]["avg_ticker_price"]
    unrealized_sold = unrealized_su * latest_price # Theoretical amount if all sold today
    
    unrealized = unrealized_sold - unrealized_spent
    
    total_returns = unrealized + realized
    tr_pct = (total_returns / buy_vol) * 100 if buy_vol > 0 else 0
    trading_period = pd.NaT
    if end_date and start_date:
        trading_period = end_date - start_date

    return (tmp_df, {
        "realized": realized,
        "unrealized": unrealized,
        "buy_vol": buy_vol,
        "sell_vol": sell_vol,
        "total_returns": total_returns,
        "tr_pct": tr_pct,
        "dark_amount": dark_amount,
        "start_date": start_date,
        "end_date": end_date,
        "trading_period": trading_period,
        "num_all_trades": len(data_list),
        "num_counted_trades": counted_trades
    })
        

tmp_df, res = calculate_success_for_df(np_df, "NVDA")

print(res)
tmp_df

{'realized': -134906.50875163433, 'unrealized': 19081991.205385398, 'buy_vol': 10125002.5, 'sell_vol': 3175001.0, 'total_returns': 18947084.696633764, 'tr_pct': 187.13165450214717, 'dark_amount': 0, 'start_date': Timestamp('2021-06-03 00:00:00'), 'end_date': Timestamp('2023-11-22 00:00:00'), 'trading_period': Timedelta('902 days 00:00:00'), 'num_all_trades': 7, 'num_counted_trades': 7}


,p_full_name,p_chamber,p_state,p_district,p_party,asset_name,ticker,ticker_location,tx_type,owner,...,notif_date,amount,asset_desc,amount_min,amount_avg,amount_max,doc_id,data_source,avg_ticker_price,is_option
19247,Nancy Pelosi,H,California,11,D,NVIDIA Corporation,NVDA,US,P,SP,...,2021-07-03,1000000-5000000,NaN,1000001.0,3000000.5,5000000.0,NaN,CAPITOL_TRADES,169.2100,False
20615,Nancy Pelosi,H,California,11,D,NVIDIA Corporation,NVDA,US,P,SP,...,2021-08-21,500000-1000000,NaN,500001.0,750000.5,1000000.0,NaN,CAPITOL_TRADES,194.7500,False
20617,Nancy Pelosi,H,California,11,D,NVIDIA Corporation,NVDA,US,P,SP,...,2021-08-21,250000-500000,NaN,250001.0,375000.5,500000.0,NaN,CAPITOL_TRADES,194.7500,False
35327,Nancy Pelosi,H,California,11,D,NVIDIA Corporation,NVDA,US,P,SP,...,2022-07-14,1000000-5000000,NaN,1000001.0,3000000.5,5000000.0,NaN,CAPITOL_TRADES,156.6150,False
36534,Nancy Pelosi,H,California,11,D,NVIDIA Corporation,NVDA,US,S,SP,...,2022-07-27,1000000-5000000,NaN,1000001.0,3000000.5,5000000.0,NaN,CAPITOL_TRADES,166.9850,False
38300,Nancy Pelosi,H,California,11,D,NVIDIA Corporation,NVDA,US,S,SP,...,2022-10-17,100000-250000,NaN,100001.0,175000.5,250000.0,NaN,CAPITOL_TRADES,129.1450,False
53052,Nancy Pelosi,H,California,11,D,NVIDIA Corporation,NVDA,US,P,SP,...,2023-12-22,1000000-5000000,NaN,1000001.0,3000000.5,5000000.0,NaN,CAPITOL_TRADES,490.1247,False


In [112]:
def calculate_success_for_politician(politician_name:str):
    mask = ticker_df["p_full_name"] == politician_name
    p_df = ticker_df[mask]
    ticker_list = p_df["ticker"].drop_duplicates().to_list()

    res_list = []
    for ticker in ticker_list:
        _, res = calculate_success_for_df(p_df, ticker)
        res["p_full_name"] = politician_name
        res["ticker"] = ticker
        res_list.append(res)
    
    tx_success_df = pd.DataFrame(res_list)

    success_summary = tx_success_df.aggregate({
        "realized": "sum",
        "unrealized": "sum",
        "total_returns": "sum",
        "buy_vol": "sum",
        "dark_amount": "sum",
        "trading_period": "max",
        "num_all_trades": "sum",
        "num_counted_trades": "sum"
    }).to_dict()
    
    total_returns = success_summary["total_returns"]
    buy_vol = success_summary["buy_vol"]
    tr_pct = total_returns * 100 / buy_vol if buy_vol > 0 else 0 
    success_summary["tr_pct"] = tr_pct

    
    
    return tx_success_df, success_summary
        
        
tx_success_df, success_summary = calculate_success_for_politician("Nancy Pelosi")
print(success_summary)

tx_success_df

{'realized': -1929503.1687932364, 'unrealized': 24377929.80799985, 'total_returns': 22448426.63920661, 'buy_vol': 33750018.5, 'dark_amount': 34475011.5, 'trading_period': Timedelta('1211 days 00:00:00'), 'num_all_trades': 73, 'num_counted_trades': 50, 'tr_pct': 66.51382024933292}


,realized,unrealized,buy_vol,sell_vol,total_returns,tr_pct,dark_amount,start_date,end_date,trading_period,num_all_trades,num_counted_trades,p_full_name,ticker
0,0.000000,0.000000e+00,0.0,0.0,0.000000e+00,0.000000,6750002.0,0,2020-08-07,NaT,4,0,Nancy Pelosi,META
1,0.000000,6.773975e+04,750000.5,0.0,6.773975e+04,9.031961,3750001.5,2021-05-21 00:00:00,2021-05-21,0 days,4,1,Nancy Pelosi,AMZN
2,0.000000,2.192452e+06,2500003.0,0.0,2.192452e+06,87.697979,6000001.0,2020-02-20 00:00:00,2023-06-15,1211 days,8,6,Nancy Pelosi,MSFT
3,0.000000,0.000000e+00,0.0,0.0,0.000000e+00,0.000000,7500002.0,0,2022-11-08,NaT,4,0,Nancy Pelosi,V
4,6443.136987,3.025285e+05,3175003.0,175000.5,3.089717e+05,9.731382,8625003.0,2021-05-21 00:00:00,2023-06-15,755 days,13,7,Nancy Pelosi,AAPL
5,0.000000,0.000000e+00,0.0,0.0,0.000000e+00,0.000000,175000.5,0,2020-05-08,NaT,1,0,Nancy Pelosi,IBKR
6,0.000000,0.000000e+00,0.0,0.0,0.000000e+00,0.000000,175000.5,0,2020-05-08,NaT,1,0,Nancy Pelosi,MORN
7,-220.546006,2.015765e+06,3925001.5,500.0,2.015545e+06,51.351444,0.0,2020-06-18 00:00:00,2022-12-20,915 days,4,4,Nancy Pelosi,CRM
8,-413330.042667,7.020060e+05,3000000.5,750001.0,2.886759e+05,9.622529,0.0,2020-06-18 00:00:00,2022-12-30,925 days,3,3,Nancy Pelosi,NFLX
9,0.000000,3.662495e+05,550001.0,0.0,3.662495e+05,66.590697,0.0,2020-06-24 00:00:00,2022-01-21,576 days,2,2,Nancy Pelosi,AXP


In [113]:
# Calculate for all
politicians_list = ticker_df["p_full_name"].drop_duplicates().to_list()

success_summary_data_list = []
tx_success_df_list = []
for p in politicians_list:
    tx_success_df, success_summary = calculate_success_for_politician(p)
    tx_success_df_list.append(tx_success_df)
    success_summary["p_full_name"] = p
    success_summary_data_list.append(success_summary)

success_summary_df = pd.DataFrame(success_summary_data_list)
tx_success_df = pd.concat(tx_success_df_list)

tx_success_df

,realized,unrealized,buy_vol,sell_vol,total_returns,tr_pct,dark_amount,start_date,end_date,trading_period,num_all_trades,num_counted_trades,p_full_name,ticker
0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,8000.5,0,2014-03-18,NaT,1,0,Susan M Collins,AIG
1,0.000000,0.000000,0.0,0.0,0.000000,0.000000,24001.5,0,2014-04-03,NaT,3,0,Susan M Collins,CRM
2,0.000000,3638.606764,8000.5,0.0,3638.606764,45.479742,0.0,2014-03-19 00:00:00,2014-03-19,0 days 00:00:00,1,1,Susan M Collins,BA
3,0.000000,149776.616328,16001.0,0.0,149776.616328,936.045349,0.0,2014-03-27 00:00:00,2014-04-03,7 days 00:00:00,2,2,Susan M Collins,MSFT
4,-492.002076,-10012.497657,24001.5,8000.5,-10504.499733,-43.766014,0.0,2014-03-27 00:00:00,2014-05-07,41 days 00:00:00,4,4,Susan M Collins,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,-254.699400,0.000000,8000.5,8000.5,-254.699400,-3.183544,0.0,2023-09-26 00:00:00,2023-10-24,28 days 00:00:00,2,2,Greg Stanton,WFC
80,261.060538,103.814356,8000.5,8000.5,364.874894,4.560651,0.0,2023-09-26 00:00:00,2023-10-24,28 days 00:00:00,2,2,Greg Stanton,DIS
81,0.000000,0.000000,0.0,0.0,0.000000,0.000000,8000.5,0,2023-10-24,NaT,1,0,Greg Stanton,PANW
0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,75000.5,0,2023-10-03,NaT,1,0,J D Vance,WMT


In [114]:
success_summary_df

,realized,unrealized,total_returns,buy_vol,dark_amount,trading_period,num_all_trades,num_counted_trades,tr_pct,p_full_name
0,-27589.888169,1.005009e+06,9.774186e+05,1782042.5,841049.5,2816 days,217,118,54.848223,Susan M Collins
1,-30179.243357,6.253095e+06,6.222916e+06,2828140.0,1907046.5,3289 days,484,391,220.035640,Sheldon Whitehouse
2,69362.436993,4.304386e+05,4.998011e+05,416026.0,368521.5,1798 days,138,95,120.136978,Jack Reed
3,-65183.598343,6.938120e+06,6.872937e+06,10946053.0,1563507.5,1732 days,147,132,62.789177,John Hoeven
4,0.000000,0.000000e+00,0.000000e+00,0.0,236506.0,NaT,12,0,0.000000,Cory A Booker
...,...,...,...,...,...,...,...,...,...,...
211,0.000000,5.853072e+05,5.853072e+05,2749064.0,80005.0,113 days,138,128,21.291144,Markwayne Mullin
212,0.000000,0.000000e+00,0.000000e+00,0.0,2475010.5,NaT,21,0,0.000000,Pete Ricketts
213,-9683.256840,4.348124e+03,-5.335133e+03,697040.5,8000.5,28 days,162,161,-0.765398,Greg Stanton
214,0.000000,0.000000e+00,0.000000e+00,0.0,75000.5,NaT,1,0,0.000000,J D Vance


In [115]:
tx_success_df = tx_success_df.sort_values(by="total_returns", ascending=False)
success_summary_df = success_summary_df.sort_values(by="total_returns", ascending=False)

tx_success_df.to_csv("tx_success.csv")
success_summary_df.to_csv("success_summary.csv")

In [116]:
def get_success_breakdown_for_politician(in_name: str):
    mask = success_summary_df["p_full_name"] == in_name
    success_summary = success_summary_df[mask].iloc[0].to_dict()

    mask = tx_success_df["p_full_name"] == in_name
    p_tx_success_df = tx_success_df[mask]

    
    return success_summary, p_tx_success_df

def get_politician_ticker_trades(p_name: str, ticker: str):
    mask = ticker_df["ticker"] == ticker
    mask = mask & (ticker_df["p_full_name"] == p_name)
    return ticker_df[mask]


In [117]:
p_name = "Nancy Pelosi"
success_summary, p_tx_success_df = get_success_breakdown_for_politician(p_name)

p_tx_success_df

,realized,unrealized,buy_vol,sell_vol,total_returns,tr_pct,dark_amount,start_date,end_date,trading_period,num_all_trades,num_counted_trades,p_full_name,ticker
15,-134906.508752,1.908199e+07,10125002.5,3175001.0,1.894708e+07,187.131655,0.0,2021-06-03 00:00:00,2023-11-22,902 days 00:00:00,7,7,Nancy Pelosi,NVDA
2,0.000000,2.192452e+06,2500003.0,0.0,2.192452e+06,87.697979,6000001.0,2020-02-20 00:00:00,2023-06-15,1211 days 00:00:00,8,6,Nancy Pelosi,MSFT
7,-220.546006,2.015765e+06,3925001.5,500.0,2.015545e+06,51.351444,0.0,2020-06-18 00:00:00,2022-12-20,915 days 00:00:00,4,4,Nancy Pelosi,CRM
10,0.000000,1.051056e+06,750000.5,0.0,1.051056e+06,140.140713,0.0,2020-09-03 00:00:00,2020-09-03,0 days 00:00:00,1,1,Nancy Pelosi,CRWD
9,0.000000,3.662495e+05,550001.0,0.0,3.662495e+05,66.590697,0.0,2020-06-24 00:00:00,2022-01-21,576 days 00:00:00,2,2,Nancy Pelosi,AXP
4,6443.136987,3.025285e+05,3175003.0,175000.5,3.089717e+05,9.731382,8625003.0,2021-05-21 00:00:00,2023-06-15,755 days 00:00:00,13,7,Nancy Pelosi,AAPL
8,-413330.042667,7.020060e+05,3000000.5,750001.0,2.886759e+05,9.622529,0.0,2020-06-18 00:00:00,2022-12-30,925 days 00:00:00,3,3,Nancy Pelosi,NFLX
1,0.000000,6.773975e+04,750000.5,0.0,6.773975e+04,9.031961,3750001.5,2021-05-21 00:00:00,2021-05-21,0 days 00:00:00,4,1,Nancy Pelosi,AMZN
0,0.000000,0.000000e+00,0.0,0.0,0.000000e+00,0.000000,6750002.0,0,2020-08-07,NaT,4,0,Nancy Pelosi,META
3,0.000000,0.000000e+00,0.0,0.0,0.000000e+00,0.000000,7500002.0,0,2022-11-08,NaT,4,0,Nancy Pelosi,V


In [120]:
"""
This is wrong,
Options aren't being accounted for correctly by capitol trades. Need to fix by scraping data
"""
t_name = "NVDA"
tmp_df = get_politician_ticker_trades(p_name, t_name)
tmp_df[["tx_date", "avg_ticker_price"]]

KeyError: "['asset_type'] not in index"